In [ ]:
import os
import numpy as np
import pandas as pd
from src import encodeMeta
from src import narrowPeak
from src import utils
from src import mobi

### 0. Parameters

In [ ]:
from mobi_parameter import Parameter
para = Parameter()

alpha_list = np.round(np.append(np.arange(0.1,1.0,0.1), np.arange(1.0,11.0,1.0)), decimals=2)
rank_list = ["RankSPP"]
rank_list.extend(["RankLinear_%.1f" % i for i in alpha_list])
rank_list.extend(["RankCrowdness"])
nbp_list = [10, 20, 40, 60, 80, 100, 120, 140, 160, 180, 200]

### 1. Input ChIP-seq data

In [ ]:
# get TF file names and TF names from metafile
metadata = encodeMeta.unique_TF_parser(para.metafile)
u_TF_files, u_TF_names, u_TF_files_w_motif, u_TF_names_w_motif, u_motif_files = metadata[5:]

# collect all ChIPped TF and pick top 3000
# these will be used for all_summit, 
# and all or subset of these will be used for downstream

TF_files_raw = ["%s/%s.bed.gz" % (para.chip_data_dir, i) for i in u_TF_files]

if os_mkdir_empty(para.TFdata_dir):
    for i in range(len(TF_files_raw)):
        narrowPeak.get_subset(
            infile=TF_files_raw[i],
            outfile="%s/%s.bed" % (para.TFdata_dir, u_TF_files[i]),
            data_proportion=para.data_n_peaks,
            sort_column=7,
            read_method="zcat")

### 2. Collect summits of all TFBS for crowding score

In [ ]:
# collect all summits from all ChIPped TF (top 3000 peaks of each bed file)
TF_files_for_summit = [para.TFdata_dir+i+".bed" for i in u_TF_files]

mobi.get_all_summit_file(
    TF_files=TF_files_for_summit, 
    all_summit_file=para.all_summit_file)

### 3. Calculate crowding score for each TFBS

In [ ]:
# Subset of all TF that will be used in the downstream analysis
# here use TFs that have known motifs (so that we could do validation)
TF_files_for_crowdness = [para.TFdata_dir+i+".bed" for i in u_TF_files_w_motif]

# compute crowdness score
mobi.get_crowdness_sliding_window(
        TF_files=TF_files_for_crowdness,
        result_dir=para.crowdness_dir,
        all_summit_file=para.all_summit_file,
        crowdness_n_sliding=para.crowdness_n_sliding)

### 4. Calculate known motif occurence

In [ ]:
TF_files_w_crowdness = [para.crowdness_dir+i+".bed" for i in u_TF_files_w_motif]
motif_fimo_files = [para.fimo_dir+i+".bed" for i in u_motif_files]

tmp_dir = utils.os_mkdir_tmp()

# all test binding site length
for nn in nbp_list: 
    os.makedirs("%s/%d" % (tmp_dir, nn))
    
    # trim binding sites to desire length
    for tf in TF_files_w_crowdness:
        narrowPeak.get_fix_width_region(
            infile=tf,
            outfile="%s/%d/%s" % (tmp_dir, nn, os.path.basename(tf)),
            width=nn,
            summit_col=None,
            keep_col=[4,5],
            read_method="cat")
    
    # compute known motif occurence
    TF_files_w_crowdness_trimmed = ["%s/%d/%s.bed" % (tmp_dir, nn, i) for i in u_TF_files_w_motif]
    mobi.get_motif_hit(
        TF_files=TF_files_w_coldness_trimmed,
        motif_fimo_files=motif_fimo_files,
        result_dir=para.motif_count_dir+str(nn))
    
    # remove TFs that fail in all previous processes
    utils.os_remove_size_zero_file(para.motif_count_dir+str(nn)+"/")

### 5. Rank TFBS by SPP/C-score/BC-score

In [ ]:
######## rank file ########
os_mkdir_empty(para.rank_file_dir)

for nn in nbp_list:
    for rank in rank_list:
        # set alpha based on the input Ranking method
        if "_" in rank:
            alpha = float(rank.split("_")[1])
        else:
            alpha = 1
        
        # generate rank file
        TF_files_w_motif_count = [para.motif_count_dir+str(nn)+"/"+i+".bed" for i in u_TF_names_w_motif]
        mobi.get_rank_file(
            rank_method=rank,
            TF_files=TF_files_w_motif_count,
            result_file="%s/%s_%s.txt" % (para.rank_file_dir, rank, nn),
            bs_half_length=nn,
            rank_linear_alpha=alpha)

### (Optional) Calculate known motif enrichment

In [ ]:
# the ranking+length combinations that will be used
rank_files = [i for i in os.listdir(para.rank_file_dir) if i.endswith(".txt") and not i.startswith(".")]

# calculate enrichment
for rank_file in rank_files:
    for n_seq in range(50,3050,50):
        mobi.get_motif_enrichment(
            rank_file=rank_file,
            n_seq=n_seq,
            result_dir=para.enrichment_dir+rank_file.replace(".txt", "/"),
            enrichment_method="EnrichmentFraction")

# summarize results
mobi.summarize_motif_enrichment(para.enrichment_dir)

### 6. Convert ranked BS bed coordinate to sequence fasta

In [ ]:
for nn in nbp_list:
    for rank in rank_list:
        # extract the input sequences for motif inference from rank files
        mobi.get_bed_from_rank_file(
            rank_file="%s/%s_%s.txt" % (para.rank_file_dir, rank, nn),
            inference_n_seq=para.inference_n_seq,
            inference_nbp=nn,
            result_dir="%s/%s_%d" % (para.bed_dir, rank, nn))
        # convert from bed format to fasta
        mobi.get_fasta_from_bed(
            bed_dir="%s/%s_%d" % (para.bed_dir, rank, nn),
            genome_file=para.genome_file,
            result_dir="%s/%s_%d" % (para.fasta_dir, rank, nn))

# delete TFs with empty fasta
for i in os.listdir(para.fasta_dir):
    if not i.startswith("."):
        utils.os_remove_size_zero_file(para.fasta_dir+i)

### 7. Motif inference

In [ ]:
# run inference
# generate joblist

for tool in ["DREME", "HOMER", "MEME"]:
    for nn in nbp_list:
        for rank in rank_list:
            mobi.run_inference_joblist(
                inference_tool=tool,
                fasta_dir="%s/%s_%d/" % (para.fasta_dir, rank, nn),
                hpc_joblist_file="/home/jg2447/log/Inference_%s_%s_%s_%d.txt" % (para.job_id, tool, rank, nn),
                inference_motif_dir="%s/%s_%s_%d/" % (para.motif_dir, tool, rank, nn),
                keep_motif_only=True)

# generate dSQ file
os.system("cat /home/jg2447/log/Inference_%s_*.txt > /home/jg2447/log/%s-inf" % (para.job_id, para.job_id))
os.system("rm /home/jg2447/log/Inference_%s_*.txt" % para.job_id)
utils.dsq_get_sh("/home/jg2447/log/%s-inf" % para.job_id)

### 8. Compare inferenced motif to known motifs

In [ ]:
# run tomtom
# generate joblist

for tool in ["DREME", "HOMER", "MEME"]:
    for nn in nbp_list:
        for rank in rank_list:
            mobi.run_tomtom_joblist(
                inference_tool=tool,
                fasta_dir="%s/%s_%d/" % (para.fasta_dir, rank, nn),
                hpc_joblist_file="/home/jg2447/log/Tomtom_%s_%s_%s_%d.txt" % (para.job_id, tool, rank, nn),
                inference_motif_dir="%s/%s_%s_%d/" % (para.motif_dir, tool, rank, nn),
                known_motif_dir=para.known_motif_dir,
                inference_tomtom_dir="%s/%s_%s_%d/" % (para.tomtom_dir, tool, rank, nn),
                keep_motif_only=True)

# generate dSQ file
os.system("cat /home/jg2447/log/Tomtom_%s_*.txt > /home/jg2447/log/%s-tom" % (para.job_id, para.job_id))
os.system("rm /home/jg2447/log/Tomtom_%s_*.txt" % para.job_id)
utils.dsq_get_sh("/home/jg2447/log/%s-tom" % para.job_id)

### 9. Calculate hits or p-value from TOMTOM result

In [ ]:
os_mkdir_empty(para.tomtom_summary_dir)

mtype="count"
for tool in ["DREME", "HOMER", "MEME"]:
    for nn in nbp_list:
        for rank in rank_list:
            hits = mobi.get_tomtom_hits(
                tomtom_dir=para.tomtom_dir,
                motif_dir=para.motif_dir,
                tool=tool, rank=rank, nn=nn, match_type=mtype,
                n_tomtom_motif=[0,1,2,3,4],
                tf_subset=None)
            pd.DataFrame(hits).transpose().to_csv(
                "%s/%s_%s_%s_%s_top5.txt" % (para.tomtom_summary_dir, tool, rank, nn, mtype),
                sep="\t", index=False, header=False, float_format='%.3f')

### 10. Find the optimal parameters (Rank and Length)

In [ ]:
with open("%s/optimal_%s.txt" % (para.result_folder, top), "w") as f:
    for tool in ["DREME", "HOMER", "MEME"]:
        df = mobi.get_tomtom_summary_data(para.tomtom_summary_dir, nbp_list, rank_list, tool, top, improve=False, tf_subset=None)
        idx = utils.df_global_max_index(df)
        f.write("%s: %s, %s\n" % (tool, idx[0], idx[1]))

### (Optional) Calculate performance evaluation statistics

In [ ]:
from sklearn.model_selection import KFold

top = "top5"

# list of TFs
TFs = pd.read_csv("%s/DREME_RankSPP_100_count_%s.txt" % (para.tomtom_summary_dir, top), sep="\t", header=None)[0].values
# collect number of known motifs for each TF
n_known_TFs = []
for tf in TFs:
    with open("%s/%s.meme" % (para.known_motif_dir, tf), "r") as f:
        mf = f.readlines()
    n_known = len([i for i in mf if i.startswith("MOTIF")])
    n_known_TFs.append(n_known)
n_known_TFs = np.array(n_known_TFs)

DREME_result = []
HOMER_result = []
MEME_result = []

# subsample and calculate the performance statistics for 10k times
for rs in range(2000):
    kf = KFold(n_splits=5, shuffle=True, random_state=rs)
    for i,j in kf.split(TFs):
        for tool in ["DREME", "HOMER", "MEME"]:
            
            # use subset i to get optimal parameters
            df = mobi.get_tomtom_summary_data(para.tomtom_summary_dir, nbp_list, rank_list, tool, top, improve=False, tf_subset=TFs[i])
            idx = utils.df_global_max_index(df)
            
            # calculate the results in subset j as testing performance
            optimal_count = pd.read_csv("%s/%s_%s_%s_count_%s.txt" % (para.tomtom_summary_dir, tool, idx[0], idx[1], top), sep="\t", header=None)
            optimal_count = optimal_count[optimal_count[0].isin(TFs[j])].sort_values(0).reset_index(drop=True).copy()

            spp_count = pd.read_csv("%s/%s_RankSPP_100_count_%s.txt" % (para.tomtom_summary_dir, tool, top), sep="\t", header=None)
            spp_count = spp_count[spp_count[0].isin(TFs[j])].sort_values(0).reset_index(drop=True).copy()

            diff = (optimal_count[1] - spp_count[1]).values

            optimal_known_hit = pd.read_csv("%s/%s_%s_%s_known_hit_%s.txt" % (para.tomtom_summary_dir, tool, idx[0], idx[1], top), sep="\t", header=None)
            optimal_known_hit = optimal_known_hit[optimal_known_hit[0].isin(TFs[j])].sort_values(0).reset_index(drop=True).copy()

            spp_known_hit = pd.read_csv("%s/%s_RankSPP_100_known_hit_%s.txt" % (para.tomtom_summary_dir, tool, top), sep="\t", header=None)
            spp_known_hit = spp_known_hit[spp_known_hit[0].isin(TFs[j])].sort_values(0).reset_index(drop=True).copy()

            diff_known = (optimal_known_hit[1] - spp_known_hit[1]).values

            result = [spp_count[1].mean(), # spp correctly inf motif
                      optimal_count[1].mean(), # optimal correctly inf motif
                      np.sum(diff > 0), # fraction of increase inf motif
                      np.sum(diff == 0), # fraction of noChange inf motif
                      np.sum(diff < 0), # fraction of decrease inf motif
                      spp_known_hit[1].mean(), # spp known motif hit
                      optimal_known_hit[1].mean(), # optimal known motif hit
                      np.sum(diff_known > 0), # fraction of increase known motif hit
                      np.sum(diff_known == 0), # fraction of noChange known motif hit
                      np.sum(diff_known < 0), # fraction of decrease known motif hit
                      (spp_known_hit[1]/n_known_TFs[j]).mean(), # spp known motif coverage percentage
                      (optimal_known_hit[1]/n_known_TFs[j]).mean()] # optimal known motif coverage percentage
            
            if tool == "DREME":
                DREME_result.append(result)
            elif tool == "HOMER":
                HOMER_result.append(result)
            elif tool == "MEME":
                MEME_result.append(result)

# save result to file
DREME_result = pd.DataFrame(DREME_result)
DREME_result.to_csv("%s/DREME_%s.txt" % (para.performance_dir, top), sep="\t", header=False, index=False, float_format="%.3f")
HOMER_result = pd.DataFrame(HOMER_result)
HOMER_result.to_csv("%s/HOMER_%s.txt" % (para.performance_dir, top), sep="\t", header=False, index=False, float_format="%.3f")
MEME_result = pd.DataFrame(MEME_result)
MEME_result.to_csv("%s/MEME_%s.txt" % (para.performance_dir, top), sep="\t", header=False, index=False, float_format="%.3f")